# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [13]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：c:\ecommerce-user-analysis-seed\data\E Commerce Dataset.xlsx
项目输出目录：c:\ecommerce-user-analysis-seed\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 在此写下你的答案：
# 1.每条记录代表一位客户（Customer），包含其人口统计信息、平台使用行为、购物偏好、满意度、投诉情况以及流失状态等特征。
# 2.目标变量是 Churn 列（1 = 已流失，0 = 未流失）。该项目旨在通过其他特征预测客户是否流失。
# 3.CustomerID 是唯一标识符，不是测量值，其数值大小没有实际含义（如 50001 与 50002 之间的大小关系无意义）。若将其当作连续数值，会导致模型学习到虚假关系，造成过拟合。在建模前应将其从特征集中移除（可保留为索引或用于后续匹配）

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量

In [3]:
# ==================== 2. 构建数据质量报告 ====================

def build_quality_report(data):
    """
    构建字段级数据质量报告。
    
    参数:
        data: DataFrame
        
    返回:
        DataFrame: 包含字段名、数据类型、缺失数量、缺失比例(%)、唯一值数量
    """
    report = pd.DataFrame({
        '数据类型': data.dtypes,
        '缺失数量': data.isnull().sum(),
        '缺失比例(%)': (data.isnull().sum() / len(data)) * 100,
        '唯一值数量': data.nunique()
    })
    # 将缺失比例保留两位小数
    report['缺失比例(%)'] = report['缺失比例(%)'].round(2)
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
print("清洗前数据质量报告：")
display(quality_before)

# 保存清洗前质量报告到CSV（用于最终交付）
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", encoding='utf-8-sig')
print(f"\n清洗前质量报告已保存至：{OUTPUT_DIR / 'data_quality_before.csv'}")

清洗前数据质量报告：


,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6



清洗前质量报告已保存至：c:\ecommerce-user-analysis-seed\output\day04_project\data_quality_before.csv


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# ==================== 初始审计 ====================

# 1. 完全重复行数
duplicate_rows = raw_df.duplicated().sum()
print(f"完全重复行数：{duplicate_rows}")

# 2. CustomerID 重复数量
customerid_duplicates = raw_df['CustomerID'].duplicated().sum()
print(f"CustomerID 重复数量：{customerid_duplicates}")

# 3. Churn 频数和流失率
churn_counts = raw_df['Churn'].value_counts()
churn_rate = (raw_df['Churn'].sum() / len(raw_df)) * 100
print("\nChurn 频数：")
print(churn_counts)
print(f"流失率：{churn_rate:.2f}%")

# 4. 主要类别字段的频数
categorical_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
print("\n主要类别字段频数：")
for col in categorical_cols:
    print(f"\n{col}：")
    print(raw_df[col].value_counts())

# 可选：也将审计结果保存到日志中（但非必须）

完全重复行数：0
CustomerID 重复数量：0

Churn 频数：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

主要类别字段频数：

PreferredLoginDevice：
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode：
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat：
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [6]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}





# ==================== 3. 数据清洗函数 ====================

def clean_data(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """
    根据项目规则清洗数据，返回清洗后的 DataFrame 和处理日志列表。
    """
    logs = []
    df_clean = df.copy()
    
    # 1. 删除完全重复行（若有）
    duplicate_rows = df_clean.duplicated().sum()
    if duplicate_rows > 0:
        df_clean = df_clean.drop_duplicates()
        logs.append(f"删除了 {duplicate_rows} 行完全重复记录。")
    else:
        logs.append("无完全重复行。")
    
    # 2. 数值字段缺失值填补（使用总体中位数）
    numeric_cols = [
        "Tenure", "WarehouseToHome", "HourSpendOnApp",
        "OrderAmountHikeFromlastYear", "CouponUsed",
        "OrderCount", "DaySinceLastOrder"
    ]
    for col in numeric_cols:
        if col in df_clean.columns:
            median_val = df_clean[col].median()
            missing_count = df_clean[col].isnull().sum()
            if missing_count > 0:
                df_clean[col] = df_clean[col].fillna(median_val)
                logs.append(f"列 '{col}' 用中位数 {median_val:.2f} 填补了 {missing_count} 个缺失值。")
            else:
                logs.append(f"列 '{col}' 无缺失值。")
    
    # 3. 类别字段统一映射
    category_mappings = {
        "PreferredLoginDevice": {"Phone": "Mobile Phone"},
        "PreferredPaymentMode": {"COD": "Cash on Delivery", "CC": "Credit Card"},
        "PreferedOrderCat": {"Mobile": "Mobile Phone"}
    }
    for col, mapping in category_mappings.items():
        if col in df_clean.columns:
            original_values = df_clean[col].copy()
            df_clean[col] = df_clean[col].replace(mapping)
            changed = (original_values != df_clean[col]).sum()
            if changed > 0:
                logs.append(f"列 '{col}' 按规则统一了 {changed} 个值。")
            else:
                logs.append(f"列 '{col}' 无需要统一的值。")
    
    return df_clean, logs

# 执行清洗
cleaned_df, cleaning_logs = clean_data(raw_df)

# 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 保存质量报告和日志
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", encoding='utf-8-sig')
pd.DataFrame({"日志": cleaning_logs}).to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding='utf-8-sig')

# 保存清洗后数据
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding='utf-8-sig')

print("清洗完成。")
print(f"清洗后数据形状：{cleaned_df.shape}")
print(f"日志条目数：{len(cleaning_logs)}")
print("所有输出已保存至：", OUTPUT_DIR)

清洗完成。
清洗后数据形状：(5630, 20)
日志条目数：11
所有输出已保存至： c:\ecommerce-user-analysis-seed\output\day04_project


---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [7]:
# ==================== 4. 可复用清洗函数 ====================

def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 1. 复制数据，避免覆盖原始数据
    df = data.copy()
    logs = []
    
    # 记录初始形状
    initial_shape = df.shape
    logs.append({
        "步骤": "初始数据",
        "处理规则": "无",
        "处理前记录数": initial_shape[0],
        "处理后记录数": initial_shape[0],
        "影响记录数": 0
    })
    
    # 2. 删除完全重复行
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        df = df.drop_duplicates()
        logs.append({
            "步骤": "删除重复行",
            "处理规则": "删除完全重复的行",
            "处理前记录数": initial_shape[0],
            "处理后记录数": df.shape[0],
            "影响记录数": duplicate_count
        })
    else:
        logs.append({
            "步骤": "删除重复行",
            "处理规则": "无完全重复行",
            "处理前记录数": df.shape[0],
            "处理后记录数": df.shape[0],
            "影响记录数": 0
        })
    
    # 3. 数值字段缺失值填补（使用总体中位数）
    numeric_cols = [
        "Tenure", "WarehouseToHome", "HourSpendOnApp",
        "OrderAmountHikeFromlastYear", "CouponUsed",
        "OrderCount", "DaySinceLastOrder"
    ]
    for col in numeric_cols:
        if col in df.columns:
            missing_before = df[col].isnull().sum()
            if missing_before > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                missing_after = df[col].isnull().sum()
                logs.append({
                    "步骤": f"填补缺失值 - {col}",
                    "处理规则": f"用中位数 {median_val:.2f} 填补",
                    "处理前记录数": df.shape[0],
                    "处理后记录数": df.shape[0],
                    "影响记录数": missing_before - missing_after
                })
            else:
                logs.append({
                    "步骤": f"检查缺失值 - {col}",
                    "处理规则": "无缺失值",
                    "处理前记录数": df.shape[0],
                    "处理后记录数": df.shape[0],
                    "影响记录数": 0
                })
    
    # 4. 类别字段统一映射
    category_mappings = {
        "PreferredLoginDevice": {"Phone": "Mobile Phone"},
        "PreferredPaymentMode": {"COD": "Cash on Delivery", "CC": "Credit Card"},
        "PreferedOrderCat": {"Mobile": "Mobile Phone"}
    }
    for col, mapping in category_mappings.items():
        if col in df.columns:
            before = df[col].copy()
            df[col] = df[col].replace(mapping)
            changed = (before != df[col]).sum()
            if changed > 0:
                logs.append({
                    "步骤": f"类别标准化 - {col}",
                    "处理规则": f"映射 {mapping}",
                    "处理前记录数": df.shape[0],
                    "处理后记录数": df.shape[0],
                    "影响记录数": changed
                })
            else:
                logs.append({
                    "步骤": f"类别标准化 - {col}",
                    "处理规则": "无需要统一的值",
                    "处理前记录数": df.shape[0],
                    "处理后记录数": df.shape[0],
                    "影响记录数": 0
                })
    
    # 5. 类型转换：Churn 和 Complain 转为整数类型
    for col in ["Churn", "Complain"]:
        if col in df.columns:
            df[col] = df[col].astype(int)
            logs.append({
                "步骤": f"类型转换 - {col}",
                "处理规则": "转为整数类型",
                "处理前记录数": df.shape[0],
                "处理后记录数": df.shape[0],
                "影响记录数": df.shape[0]  # 所有记录都受影响
            })
    
    # 6. 生成日志 DataFrame
    cleaning_log = pd.DataFrame(logs)
    
    return df, cleaning_log


# 测试清洗函数
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

# 显示清洗日志
print("清洗日志：")
display(cleaning_log)

# 保存清洗后数据和日志
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding='utf-8-sig')
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding='utf-8-sig')

print(f"\n清洗后数据形状：{cleaned_df.shape}")
print(f"日志已保存，共 {len(cleaning_log)} 条记录。")
print("所有输出已保存至：", OUTPUT_DIR)

清洗日志：


,步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,初始数据,无,5630,5630,0
1,删除重复行,无完全重复行,5630,5630,0
2,填补缺失值 - Tenure,用中位数 9.00 填补,5630,5630,264
3,填补缺失值 - WarehouseToHome,用中位数 14.00 填补,5630,5630,251
4,填补缺失值 - HourSpendOnApp,用中位数 3.00 填补,5630,5630,255
5,填补缺失值 - OrderAmountHikeFromlastYear,用中位数 15.00 填补,5630,5630,265
6,填补缺失值 - CouponUsed,用中位数 1.00 填补,5630,5630,256
7,填补缺失值 - OrderCount,用中位数 2.00 填补,5630,5630,258
8,填补缺失值 - DaySinceLastOrder,用中位数 3.00 填补,5630,5630,307
9,类别标准化 - PreferredLoginDevice,映射 {'Phone': 'Mobile Phone'},5630,5630,1231



清洗后数据形状：(5630, 20)
日志已保存，共 14 条记录。
所有输出已保存至： c:\ecommerce-user-analysis-seed\output\day04_project


### 任务 3：运行清洗函数并查看日志

In [8]:
# ==================== 任务 3：运行清洗函数并查看日志 ====================

# 执行清洗（如果尚未执行，取消下面的注释）
# cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

# 显示清洗日志
print("清洗日志：")
display(cleaning_log)

# 显示清洗后数据前5行
print("\n清洗后数据前5行：")
display(cleaned_df.head())

# 检查数据类型转换是否成功
print("\n数据类型检查（Churn 和 Complain）：")
print(cleaned_df[['Churn', 'Complain']].dtypes)

# 检查缺失值是否已处理（数值列）
numeric_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 
                'OrderAmountHikeFromlastYear', 'CouponUsed', 
                'OrderCount', 'DaySinceLastOrder']
print("\n数值列缺失值检查（应全为0）：")
print(cleaned_df[numeric_cols].isnull().sum())

# 检查类别标准化是否生效
print("\n类别标准化检查：")
print("PreferredLoginDevice 唯一值：", cleaned_df['PreferredLoginDevice'].unique())
print("PreferredPaymentMode 唯一值：", cleaned_df['PreferredPaymentMode'].unique())
print("PreferedOrderCat 唯一值：", cleaned_df['PreferedOrderCat'].unique())

清洗日志：


,步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,初始数据,无,5630,5630,0
1,删除重复行,无完全重复行,5630,5630,0
2,填补缺失值 - Tenure,用中位数 9.00 填补,5630,5630,264
3,填补缺失值 - WarehouseToHome,用中位数 14.00 填补,5630,5630,251
4,填补缺失值 - HourSpendOnApp,用中位数 3.00 填补,5630,5630,255
5,填补缺失值 - OrderAmountHikeFromlastYear,用中位数 15.00 填补,5630,5630,265
6,填补缺失值 - CouponUsed,用中位数 1.00 填补,5630,5630,256
7,填补缺失值 - OrderCount,用中位数 2.00 填补,5630,5630,258
8,填补缺失值 - DaySinceLastOrder,用中位数 3.00 填补,5630,5630,307
9,类别标准化 - PreferredLoginDevice,映射 {'Phone': 'Mobile Phone'},5630,5630,1231



清洗后数据前5行：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60



数据类型检查（Churn 和 Complain）：
Churn       int64
Complain    int64
dtype: object

数值列缺失值检查（应全为0）：
Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
dtype: int64

类别标准化检查：
PreferredLoginDevice 唯一值： ['Mobile Phone' 'Computer']
PreferredPaymentMode 唯一值： ['Debit Card' 'UPI' 'Credit Card' 'Cash on Delivery' 'E wallet']
PreferedOrderCat 唯一值： ['Laptop & Accessory' 'Mobile Phone' 'Others' 'Fashion' 'Grocery']


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [9]:
# ==================== 5. 数据转换与候选异常值检查 ====================

# 确保 cleaned_df 已存在（如果尚未执行清洗，先运行清洗代码）
# 如果已执行清洗，直接使用 cleaned_df

# 定义 IQR 摘要函数
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO 1：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [0, 6, 12, 24, 48, float('inf')]
tenure_labels = ['0-6个月', '6-12个月', '12-24个月', '24-48个月', '48个月以上']
cleaned_df['TenureGroup'] = pd.cut(cleaned_df['Tenure'], bins=tenure_bins, labels=tenure_labels, right=False)

# TODO 2：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df['IsMobileLogin'] = (cleaned_df['PreferredLoginDevice'] == 'Mobile Phone').astype(int)

# TODO 3：生成 outlier_report（每行对应一个待检查字段）
outlier_cols = ['WarehouseToHome', 'OrderCount', 'CashbackAmount']
outlier_data = []
for col in outlier_cols:
    stats = iqr_outlier_summary(cleaned_df[col])
    outlier_data.append({
        '字段': col,
        'Q1': stats['Q1'],
        'Q3': stats['Q3'],
        '下限': stats['下限'],
        '上限': stats['上限'],
        '候选异常值数量': stats['候选异常值数量']
    })
outlier_report = pd.DataFrame(outlier_data)

# 显示结果
print("TenureGroup 分布：")
print(cleaned_df['TenureGroup'].value_counts().sort_index())
print("\nIsMobileLogin 分布：")
print(cleaned_df['IsMobileLogin'].value_counts())
print("\n候选异常值报告：")
display(outlier_report)

# 保存更新后的数据和异常值报告
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding='utf-8-sig')
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding='utf-8-sig')

print(f"\n派生特征已添加，数据形状：{cleaned_df.shape}")
print(f"异常值报告已保存至：{OUTPUT_DIR / 'outlier_report.csv'}")

TenureGroup 分布：
TenureGroup
0-6个月      1967
6-12个月     1585
12-24个月    1574
24-48个月     500
48个月以上        4
Name: count, dtype: int64

IsMobileLogin 分布：
IsMobileLogin
1    3996
0    1634
Name: count, dtype: int64

候选异常值报告：


,字段,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438



派生特征已添加，数据形状：(5630, 22)
异常值报告已保存至：c:\ecommerce-user-analysis-seed\output\day04_project\outlier_report.csv


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [10]:
# TODO：完成业务规则检查
rule_checks = {
    "使用时长小于 0": "Tenure < 0",
    "仓库距离小于 0": "WarehouseToHome < 0",
    "订单数小于或等于 0": "OrderCount <= 0",
    "返现金额小于 0": "CashbackAmount < 0"
}

violation_counts = {}
for rule_name, query_str in rule_checks.items():
    count = cleaned_df.query(query_str).shape[0]
    violation_counts[rule_name] = count

business_rule_report = pd.DataFrame({
    "规则": list(violation_counts.keys()),
    "不合规记录数": list(violation_counts.values())
})
display(business_rule_report)

# 处理结论
print("""
处理结论：
所有检查规则的不合规记录数均为 0，说明数据在以下方面不存在业务异常：
- 使用时长（Tenure）均 ≥ 0
- 仓库距离（WarehouseToHome）均 ≥ 0
- 订单数（OrderCount）均 > 0
- 返现金额（CashbackAmount）均 ≥ 0

数据已通过基本业务合理性验证，无需对上述字段进行额外修正。
""")

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



处理结论：
所有检查规则的不合规记录数均为 0，说明数据在以下方面不存在业务异常：
- 使用时长（Tenure）均 ≥ 0
- 仓库距离（WarehouseToHome）均 ≥ 0
- 订单数（OrderCount）均 > 0
- 返现金额（CashbackAmount）均 ≥ 0

数据已通过基本业务合理性验证，无需对上述字段进行额外修正。



---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [12]:
# ==================== 6. 项目验收与交付 ====================

# 1. 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 2. 验证清洗结果是否符合预期
# 2.1 数值列不应有缺失
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值列仍有缺失值"

# 2.2 类别标准化验证
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "Phone 未统一为 Mobile Phone"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "COD 未统一为 Cash on Delivery"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "CC 未统一为 Credit Card"
assert "Mobile" not in cleaned_df["PreferedOrderCat"].unique(), "Mobile 未统一为 Mobile Phone"

# 2.3 派生特征应存在
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "派生特征缺失"

print("✅ 所有校验通过！")

# 3. 比较清洗前后缺失值变化
missing_comparison = pd.DataFrame({
    '清洗前缺失数量': quality_before['缺失数量'],
    '清洗后缺失数量': quality_after['缺失数量'],
    '减少缺失数量': quality_before['缺失数量'] - quality_after['缺失数量']
})
print("\n清洗前后缺失值对比（仅显示有变化的字段）：")
display(missing_comparison[missing_comparison['减少缺失数量'] > 0])

# 4. 导出全部交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

# 5. 输出所有交付文件路径
print("\n交付文件已导出至：", OUTPUT_DIR)
print("-" * 40)
print("1. data_quality_before.csv      (清洗前质量报告)")
print("2. data_quality_after.csv       (清洗后质量报告)")
print("3. cleaning_log.csv             (处理日志)")
print("4. ecommerce_customer_cleaned.csv (清洗后用户数据)")
print("5. outlier_report.csv           (候选异常值报告)")
print("6. business_rule_report.csv     (业务规则检查报告)")
print("-" * 40)

# 简要总结
print("\n项目完成情况：")
print(f"- 原始数据形状：{raw_df.shape}")
print(f"- 清洗后数据形状：{cleaned_df.shape}")
print(f"- 处理日志条目数：{len(cleaning_log)}")
print(f"- 数值列缺失已全部填补 (0 缺失)")
print(f"- 类别值已按规则统一")
print(f"- 新增派生特征：TenureGroup, IsMobileLogin")
print(f"- 候选异常值已识别但未删除，记录在 outlier_report.csv 中")
print(f"- 业务规则检查全部通过，无不合规记录")

print("\n✅ 项目验收通过，所有交付物已准备就绪。")

✅ 所有校验通过！

清洗前后缺失值对比（仅显示有变化的字段）：


,清洗前缺失数量,清洗后缺失数量,减少缺失数量
CouponUsed,256.00,0,256.00
DaySinceLastOrder,307.00,0,307.00
HourSpendOnApp,255.00,0,255.00
OrderAmountHikeFromlastYear,265.00,0,265.00
OrderCount,258.00,0,258.00
Tenure,264.00,0,264.00
WarehouseToHome,251.00,0,251.00



交付文件已导出至： c:\ecommerce-user-analysis-seed\output\day04_project
----------------------------------------
1. data_quality_before.csv      (清洗前质量报告)
2. data_quality_after.csv       (清洗后质量报告)
3. cleaning_log.csv             (处理日志)
4. ecommerce_customer_cleaned.csv (清洗后用户数据)
5. outlier_report.csv           (候选异常值报告)
6. business_rule_report.csv     (业务规则检查报告)
----------------------------------------

项目完成情况：
- 原始数据形状：(5630, 20)
- 清洗后数据形状：(5630, 22)
- 处理日志条目数：14
- 数值列缺失已全部填补 (0 缺失)
- 类别值已按规则统一
- 新增派生特征：TenureGroup, IsMobileLogin
- 候选异常值已识别但未删除，记录在 outlier_report.csv 中
- 业务规则检查全部通过，无不合规记录

✅ 项目验收通过，所有交付物已准备就绪。


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
数据质量问题：数值列存在缺失（Tenure等7列，缺失率4%~5%）；类别值不一致（Phone/Mobile Phone、COD/Cash on Delivery、CC/Credit Card、Mobile/Mobile Phone）；WarehouseToHome、OrderCount、CashbackAmount存在候选异常值。
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
缺失值：使用总体中位数填补，稳健且不将缺失误解为0；
类别不一致：通过映射字典统一为标准值；
候选异常值：仅记录不删除，待业务确认。
3. 为什么清洗后的数据可以作为第五天分析的输入？
数据已无缺失、类别统一、类型正确、业务规则通过，且新增了时长分层和移动端标识两个衍生特征，适合后续建模或分析。
4. 哪些处理规则仍需要业务人员确认？
中位数填补是否适合所有数值列（如Tenure按用户群体可能分化）；类别统一是否完全符合业务理解；异常值的处理阈值是否合理（如距离过远的订单是否应排除）。

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。